# RVL-CDIP SQL queries + recreation sampling

Queryable walkthrough over the local
[aharley/rvl_cdip](https://huggingface.co/datasets/aharley/rvl_cdip) SQLite index
(`.venv/rvl_cdip/rvl_cdip.db`), then a **seeded stratified draw** of **60-70
documents from each of the 16 classes** (label ids `0`-`15`) for the recreation
experiment.

**This notebook completes the sampling task:** it writes a JSONL sample set,
a machine-readable manifest, and a per-class count table under
`data/notebook_demo/rvl_cdip_recreation/`, then downloads the image archive
(~38 GB) and materializes the sampled TIFFs under `.venv/rvl_cdip/`.

> **Statistical validity**
>
> Samples are drawn **only from the official `test` split** (`SPLIT_FILTER="test"`).
> Mixing train / validation / test would leak held-out labels into the recreation
> set and break comparability with published RVL-CDIP protocols. Keep train/val
> for fitting or tuning elsewhere; evaluate on this test-only draw.

> **Prerequisites**
>
> ```bash
> python -m src.rvl_cdip build   # ~17 MB labels -> ~400k SQL rows (idempotent)
> # Free disk: recommend ≥ 45 GB before the image archive step
> ```
>
> Section **2b** downloads `rvl-cdip.tar.gz` (~38 GB) into `.venv/rvl_cdip/`
> (dual opt-in flags enabled for the full pipeline). Section **5b** selectively
> extracts the sampled images from that archive.

Related: [RVL-CDIP SQL Index](../rvl_cdip_sql.md) · CLI `python -m src.rvl_cdip --help`

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import random
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import JSON, Markdown, display

CWD = Path.cwd().resolve()
# Walk up from docs/notebooks/ (or notebooks/) until pyproject.toml is found.
REPO_ROOT = next(
    (p for p in (CWD, *CWD.parents) if (p / "pyproject.toml").exists()),
    None,
)
assert REPO_ROOT is not None, f"Could not find repo root from {CWD}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.rvl_cdip import LABEL_NAMES, RvlCdipStore, default_db_path
from src.rvl_cdip.download import (
    download_images,
    format_image_download_preflight,
    image_download_preflight,
)
from src.rvl_cdip.paths import archive_path
from src.rvl_cdip.sample_images import materialize_samples, summarize_materialize
from src.utils.io import write_json, write_jsonl

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

pd.set_option("display.max_colwidth", 72)
pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 40)

DEMO = REPO_ROOT / "data" / "notebook_demo" / "rvl_cdip_recreation"
DEMO.mkdir(parents=True, exist_ok=True)

# --- Recreation knobs ---
SEED = 42
N_PER_CLASS = 65  # choose any integer in [60, 70]
assert 60 <= N_PER_CLASS <= 70, "N_PER_CLASS must be between 60 and 70 inclusive"
# Held-out eval only: never mix train/validation into the recreation set.
SPLIT_FILTER: str = "test"
assert SPLIT_FILTER in {"train", "validation", "test"}, SPLIT_FILTER

# --- Full pipeline: ~38 GB image archive (writes ONLY under .venv/rvl_cdip) ---
I_UNDERSTAND_LARGE_DOWNLOAD = True
CONFIRM_WRITES_UNDER_VENV = True
FORCE_REDOWNLOAD = False
MATERIALIZE_SAMPLE_IMAGES = True  # selective extract after sampling

print(f"repo:     {REPO_ROOT}")
print(f"db:       {default_db_path()}")
print(f"demo out: {DEMO}")
print(f"SEED={SEED}  N_PER_CLASS={N_PER_CLASS}  SPLIT_FILTER={SPLIT_FILTER!r}")
print(
    f"archive:  {archive_path()}  present={archive_path().is_file()}  "
    f"download={I_UNDERSTAND_LARGE_DOWNLOAD and CONFIRM_WRITES_UNDER_VENV}"
)
display(
    pd.DataFrame(
        {"label_id": range(len(LABEL_NAMES)), "label": list(LABEL_NAMES)}
    ).set_index("label_id")
)

## 2. Open the SQL index (build if empty)

`RvlCdipStore()` opens `.venv/rvl_cdip/rvl_cdip.db`. If the documents table is
empty, `build_from_labels()` downloads the split label files (~17 MB) and
ingests all ~400k rows.

In [ ]:
store = RvlCdipStore()
summary = store.summary()
print(f"documents indexed: {summary['documents']:,}")
print(f"with image paths:  {summary['with_image_abspath']:,}")
print(f"built_at:          {summary.get('built_at')}")
print(f"dataset_id:        {summary.get('dataset_id')}")

if summary["documents"] == 0:
    print("Index empty - building from Hub label files (safe; no 38 GB archive)...")
    stats = store.build_from_labels()
    summary = store.summary()
    print("build stats:", stats)
    print(f"documents indexed: {summary['documents']:,}")

assert summary["documents"] > 0, (
    "RVL-CDIP index is empty. Run: python -m src.rvl_cdip build"
)

by_split_df = pd.DataFrame(summary["by_split"]).set_index("split")
by_label_df = (
    pd.DataFrame(summary["by_label"])
    .sort_values("label_id")
    .set_index("label_id")
)
display(Markdown("### Rows per split"))
display(by_split_df)
display(Markdown("### Rows per class"))
display(by_label_df)

## 2b. Download image archive (~38 GB) into `.venv/rvl_cdip` only

Hub cache + `rvl-cdip.tar.gz` write **only** under `.venv/rvl_cdip/` (never `data/` or
`~/.cache`). Setup enables both opt-in flags for the **full** recreation pipeline.
Re-run preflight first; the download is idempotent if the archive is already present.

```bash
python -m src.rvl_cdip download-images --preflight
python -m src.rvl_cdip download-images \
  --i-understand-large-download --confirm-writes-under-venv
```

In [ ]:
preflight = image_download_preflight(force=FORCE_REDOWNLOAD)
print(format_image_download_preflight(preflight))
display(preflight)

if not (I_UNDERSTAND_LARGE_DOWNLOAD and CONFIRM_WRITES_UNDER_VENV):
    raise RuntimeError(
        "Full pipeline requires I_UNDERSTAND_LARGE_DOWNLOAD=True and "
        "CONFIRM_WRITES_UNDER_VENV=True (see Setup knobs)."
    )

if not preflight["enough_free_space"] and not preflight["archive_already_present"]:
    raise RuntimeError(
        f"Need ≥ {preflight['min_free_gb']} GB free; have {preflight['free_gb']} GB."
    )

result = download_images(
    force=FORCE_REDOWNLOAD,
    i_understand_large_download=True,
    confirm_writes_under_venv=True,
)
store.record_download(result)
updated = store.refresh_image_paths()
display(
    {
        "local_path": str(result.local_path),
        "bytes": result.bytes,
        "skipped": result.skipped,
        "image_abspath_updated": updated,
        "writes_only_under": result.detail.get("writes_only_under"),
        "archive_present": archive_path().is_file(),
    }
)
assert archive_path().is_file(), "Image archive missing after download step"
print("OK image archive ready under .venv/rvl_cdip/")

## 3. Showcase SQL queries

### 3a. Class catalog + counts

`store.labels()` and ad-hoc `store.query(...)` (SELECT-only).

In [ ]:
labels_df = pd.DataFrame(store.labels()).set_index("label_id")
display(Markdown("### Label catalog"))
display(labels_df)

by_label = store.query(
    """
    SELECT d.label_id, l.name AS label, COUNT(*) AS n
    FROM documents d
    JOIN labels l ON l.label_id = d.label_id
    GROUP BY d.label_id
    ORDER BY d.label_id
    """
)
plot_df = pd.DataFrame(by_label).sort_values("label_id")
display(Markdown("### SQL count by class"))
display(plot_df.set_index("label_id"))

fig, ax = plt.subplots(figsize=(8, 6))
y_labels = [f"{row.label_id:>2d}  {row.label}" for row in plot_df.itertuples()]
ax.barh(y_labels, plot_df["n"], color="#2a6f97")
ax.invert_yaxis()
ax.set_title("RVL-CDIP documents per class")
ax.set_xlabel("count")
ax.set_ylabel("label_id / name")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v):,}"))
plt.tight_layout()
plt.show()

### 3b. Split x label table

Useful to see whether a class is balanced across train / test / validation.

In [ ]:
split_label = store.query(
    """
    SELECT d.split, d.label_id, l.name AS label, COUNT(*) AS n
    FROM documents d
    JOIN labels l ON l.label_id = d.label_id
    GROUP BY d.split, d.label_id
    ORDER BY d.split, d.label_id
    """,
    max_rows=200,
)
pivot = (
    pd.DataFrame(split_label)
    .pivot(index=["label_id", "label"], columns="split", values="n")
    .fillna(0)
    .astype(int)
)
for col in ("train", "validation", "test"):
    if col not in pivot.columns:
        pivot[col] = 0
pivot = pivot[["train", "validation", "test"]]
display(Markdown("### Split x label counts"))
display(pivot)

display(Markdown("### Example: five train invoices (`list_documents`)"))
invoice_train = store.list_documents(split="train", label="invoice", limit=5)
display(pd.DataFrame(invoice_train))

### 3c. Example filtered SELECTs

Same patterns as the CLI:

```bash
python -m src.rvl_cdip query "SELECT split, COUNT(*) AS n FROM documents GROUP BY split"
```

In [ ]:
examples = {
    "rows_per_split": """
        SELECT split, COUNT(*) AS n
        FROM documents
        GROUP BY split
        ORDER BY split
    """,
    "handwritten_train_sample": """
        SELECT d.document_id, d.split, d.label_id, l.name AS label, d.image_relpath
        FROM documents d
        JOIN labels l ON l.label_id = d.label_id
        WHERE d.label_id = 3 AND d.split = 'train'
        ORDER BY d.source_line
        LIMIT 5
    """,
    "memo_vs_letter": """
        SELECT l.name AS label, d.split, COUNT(*) AS n
        FROM documents d
        JOIN labels l ON l.label_id = d.label_id
        WHERE d.label_id IN (0, 15)
        GROUP BY l.name, d.split
        ORDER BY l.name, d.split
    """,
}

for name, sql in examples.items():
    display(Markdown(f"#### `{name}`"))
    display(pd.DataFrame(store.query(sql)))

## 4. Recreation sample — 60–70 random docs per class (0–15)

For each `label_id` in `0..15`:

1. Pull candidate rows with SQL (`WHERE label_id = ? AND split = 'test'`)
2. Draw `N_PER_CLASS` with a **seeded** `random.Random(SEED)` (reproducible;
   SQLite `ORDER BY RANDOM()` is not seedable)
3. Tag rows with seed / n_per_class / split_filter metadata

`SPLIT_FILTER="test"` keeps the recreation set on the official held-out split
(statistical best practice for evaluation / recreation scoring).

Default `N_PER_CLASS = 65` (midpoint of 60–70). Change the knob in Setup and
re-run from here.

In [ ]:
def fetch_candidates(label_id: int, split: str | None = None) -> list[dict]:
    """All indexed rows for one class (optionally one split)."""
    if split:
        sql = """
            SELECT d.document_id, d.split, d.label_id, l.name AS label,
                   d.image_relpath, d.image_abspath, d.source_line
            FROM documents d
            JOIN labels l ON l.label_id = d.label_id
            WHERE d.label_id = ? AND d.split = ?
            """
        params: tuple = (label_id, split)
    else:
        sql = """
            SELECT d.document_id, d.split, d.label_id, l.name AS label,
                   d.image_relpath, d.image_abspath, d.source_line
            FROM documents d
            JOIN labels l ON l.label_id = d.label_id
            WHERE d.label_id = ?
            """
        params = (label_id,)
    # One class ~25k rows - raise max_rows past the default 1000.
    return store.query(sql, params, max_rows=50_000)


def sample_recreation_set(
    *,
    n_per_class: int,
    seed: int,
    split: str | None = None,
) -> list[dict]:
    rng = random.Random(seed)
    out: list[dict] = []
    for label_id in range(len(LABEL_NAMES)):
        candidates = fetch_candidates(label_id, split=split)
        if len(candidates) < n_per_class:
            raise RuntimeError(
                f"label_id={label_id} ({LABEL_NAMES[label_id]!r}) has only "
                f"{len(candidates)} candidates; need {n_per_class}"
            )
        picked = rng.sample(candidates, n_per_class)
        for row in picked:
            item = dict(row)
            item["recreation_seed"] = seed
            item["n_per_class"] = n_per_class
            item["split_filter"] = split
            out.append(item)
    return out


samples = sample_recreation_set(
    n_per_class=N_PER_CLASS, seed=SEED, split=SPLIT_FILTER
)
sample_df = pd.DataFrame(samples)

print(f"total samples: {len(sample_df):,}  (expect {16 * N_PER_CLASS:,})")
counts = (
    sample_df.groupby(["label_id", "label"], sort=True)
    .size()
    .rename("n")
    .reset_index()
)
display(Markdown("### Sampled counts (must equal N_PER_CLASS for every class)"))
display(counts.set_index("label_id"))

split_counts = sample_df["split"].value_counts().rename("n").to_frame()
display(Markdown("### Sampled rows by source split (expect only test)"))
display(split_counts)

display(Markdown("### Preview (first 8 rows)"))
display(
    sample_df[
        ["label_id", "label", "split", "document_id", "image_relpath"]
    ].head(8)
)

# Hard checks - notebook "completes" only if these pass
assert len(sample_df) == 16 * N_PER_CLASS
assert set(sample_df["label_id"]) == set(range(16))
assert (counts["n"] == N_PER_CLASS).all()
assert sample_df["document_id"].is_unique
assert set(sample_df["split"]) == {SPLIT_FILTER}, (
    f"Expected all rows from {SPLIT_FILTER!r}; got {set(sample_df['split'])}"
)
print(
    f"OK stratified sample: exactly {N_PER_CLASS} unique docs × 16 classes "
    f"from split={SPLIT_FILTER!r}"
)

## 5. Export artifacts for the recreation experiment

Writes under `data/notebook_demo/rvl_cdip_recreation/`:

| File | Contents |
|------|----------|
| `recreation_samples.jsonl` | One sampled document per line |
| `recreation_manifest.json` | Seed, n_per_class, counts, db path |
| `recreation_counts.csv` | Per-class counts |

In [ ]:
samples_path = DEMO / "recreation_samples.jsonl"
manifest_path = DEMO / "recreation_manifest.json"
counts_path = DEMO / "recreation_counts.csv"

write_jsonl(samples_path, samples)
counts.to_csv(counts_path, index=False)

manifest = {
    "experiment": "rvl_cdip_recreation",
    "dataset_id": summary.get("dataset_id"),
    "db_path": summary.get("db_path"),
    "seed": SEED,
    "n_per_class": N_PER_CLASS,
    "n_classes": 16,
    "total_samples": len(samples),
    "split_filter": SPLIT_FILTER,
    "label_names": list(LABEL_NAMES),
    "counts_by_label_id": {
        int(r["label_id"]): int(r["n"]) for _, r in counts.iterrows()
    },
    "archive_path": str(archive_path()),
    "archive_present": archive_path().is_file(),
    "materialize_sample_images": MATERIALIZE_SAMPLE_IMAGES,
    "samples_path": str(samples_path.relative_to(REPO_ROOT)),
    "counts_path": str(counts_path.relative_to(REPO_ROOT)),
}
write_json(manifest_path, manifest)

print("wrote:", samples_path.relative_to(REPO_ROOT))
print("wrote:", manifest_path.relative_to(REPO_ROOT))
print("wrote:", counts_path.relative_to(REPO_ROOT))
display(Markdown("### Manifest"))
display(JSON(manifest))

# Reload check
reloaded = [
    json.loads(line)
    for line in samples_path.read_text().splitlines()
    if line.strip()
]
assert len(reloaded) == len(samples)
assert Counter(r["label_id"] for r in reloaded) == Counter(
    {i: N_PER_CLASS for i in range(16)}
)
assert all(r["split"] == SPLIT_FILTER for r in reloaded)
print("OK export verified on disk")

## 5b. Materialize sampled images from the archive

Selectively extracts each sampled TIFF from `rvl-cdip.tar.gz` into
`.venv/rvl_cdip/source/images/` (does **not** unpack the full 38 GB tree).
Requires section **2b** to have completed successfully.

In [ ]:
if not MATERIALIZE_SAMPLE_IMAGES:
    raise RuntimeError("Set MATERIALIZE_SAMPLE_IMAGES=True for the full pipeline.")

assert archive_path().is_file(), (
    "Missing rvl-cdip.tar.gz — re-run section 2b (image archive download)."
)

mats = materialize_samples(samples, run_ocr=False)
mat_summary = summarize_materialize(mats)
display(Markdown("### Materialize summary"))
display(mat_summary)
mat_df = pd.DataFrame([m.to_dict() for m in mats])
display(mat_df.head(8))

n_ok = int(mat_summary["n_with_image"])
assert n_ok == len(samples), (
    f"Expected images for all {len(samples)} samples; got {n_ok}. "
    f"errors={mat_summary.get('n_errors')}"
)

# Persist resolved paths back onto the export for downstream notebooks
path_by_id = {m.document_id: m for m in mats}
enriched = []
for row in samples:
    item = dict(row)
    mat = path_by_id.get(str(item["document_id"]))
    if mat and mat.image_path is not None:
        item["image_abspath"] = str(mat.image_path)
    enriched.append(item)
write_jsonl(samples_path, enriched)
samples[:] = enriched
sample_df = pd.DataFrame(samples)
updated = store.refresh_image_paths()
print(f"OK materialized {n_ok} images; refresh_image_paths updated {updated} rows")
print("rewrote:", samples_path.relative_to(REPO_ROOT))

## 6. Peek + next steps

One representative row per class (all from `test`), with resolved image paths
after materialize. Downstream: `rvl_cdip_openrouter_eval.ipynb`.

In [ ]:
peek = (
    sample_df.sort_values(["label_id", "document_id"])
    .groupby("label_id", as_index=False)
    .first()[["label_id", "label", "split", "document_id", "image_relpath", "image_abspath"]]
)
display(Markdown("### One sample per class"))
display(peek.set_index("label_id"))

example_id = str(samples[0]["document_id"])
display(Markdown(f"### store.get_document({example_id!r})"))
display(JSON(store.get_document(example_id) or {}))

rel_samples = samples_path.relative_to(REPO_ROOT)
rel_manifest = manifest_path.relative_to(REPO_ROOT)
n_with_abs = int(sample_df["image_abspath"].notna().sum()) if "image_abspath" in sample_df else 0
display(
    Markdown(
        f"**Done.** Recreation set ready (split=`{SPLIT_FILTER}`):\n\n"
        f"- `{rel_samples}` — **{len(samples)}** rows\n"
        f"- `{rel_manifest}` — `seed={SEED}`, `n_per_class={N_PER_CLASS}`\n"
        f"- archive `{archive_path().name}` present; "
        f"**{n_with_abs}** samples have `image_abspath`\n"
    )
)
print("CLI equivalents:")
print("  python -m src.rvl_cdip summary")
print(
    '  python -m src.rvl_cdip query '
    '"SELECT label_id, COUNT(*) AS n FROM documents GROUP BY label_id"'
)